# TinyDoc-VLM: Colab Training (T4)

Auto-trains all 3 stages. Run cells 1-4 in order; disconnect-safe.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/tinydoc-vlm'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
print(f'Drive ready at {DRIVE_ROOT}')

## 2. Clone repo & install deps

In [ ]:
import os, sys
REPO_DIR = '/content/tinydoc-vlm'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/eulogik/TinyDoc-VLM.git {REPO_DIR}

%cd {REPO_DIR}
!git pull
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers sentencepiece tokenizers pillow numpy pandas tqdm pyyaml gradio optimum onnx onnxruntime einops faker jinja2 pydantic
!pip install -q accelerate
print('Deps installed')

## 3. Restore data from Drive

Force-clean restore (rmtree + fresh copy) to avoid partial/incomplete data.

In [ ]:
import shutil
from pathlib import Path

DATA_DIR = Path('data/synthetic')
DRIVE_DATA = Path(f'{DRIVE_ROOT}/data/synthetic')

drive_has_data = (DRIVE_DATA / 'output' / 'manifest.jsonl').exists()
drive_has_images = drive_has_data and len(list((DRIVE_DATA / 'output' / 'images').glob('*.png'))) > 100

if drive_has_images:
    print('Restoring data from Drive...')
    if DATA_DIR.exists():
        shutil.rmtree(str(DATA_DIR))
    shutil.copytree(str(DRIVE_DATA), str(DATA_DIR))
    n = sum(1 for _ in open(DATA_DIR / 'output' / 'manifest.jsonl'))
    m = len(list((DATA_DIR / 'output' / 'images').glob('*.png')))
    print(f'Done: {n} docs, {m} images')
elif DATA_DIR.exists() and len(list((DATA_DIR / 'output' / 'images').glob('*.png'))) > 100:
    n = sum(1 for _ in open(DATA_DIR / 'output' / 'manifest.jsonl'))
    print(f'Data already complete locally ({n} docs) — skipping copy')
else:
    print('No cached data. Generating 1000 docs (~5 min)...')
    !python data/synthetic/generator.py --num-docs 1000 --output-dir data/synthetic/output
    print('Saving to Drive...')
    DRIVE_DATA.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(DATA_DIR), str(DRIVE_DATA), dirs_exist_ok=True)

n = sum(1 for _ in open(DATA_DIR / 'output' / 'manifest.jsonl'))
m = len(list((DATA_DIR / 'output' / 'images').glob('*.png')))
assert m > 100, f'Only {m} images — data restore failed'
print(f'Ready: {n} docs, {m} images')

## 4. Train all 3 stages (auto-detect & auto-advance)

- Saves to **local SSD** (`/content/checkpoints/`) — no Drive quota hit during training.
- Syncs final checkpoint to Drive at each stage end (overwrites, no duplicates).
- Resumes partial stages from local or Drive.
- Completed stages are skipped.

In [ ]:
import yaml
import torch
import shutil
import logging
from torch.utils.data import random_split
from pathlib import Path

from tinydoc_vlm import (
    TinyDocVLMConfig,
    TinyDocVLMForConditionalGeneration,
    TinyDocVLMProcessor,
    TinyDocVLMTrainer,
    TrainerConfig,
    DocumentDataset,
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('colab')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f'Device: {device}')

# --- Clean up old checkpoint cruft from Drive (free space) ---
for d in Path(f'{DRIVE_ROOT}/checkpoints').rglob('step_*'):
    shutil.rmtree(d, ignore_errors=True)
for d in Path(f'{DRIVE_ROOT}/checkpoints').rglob('init'):
    shutil.rmtree(d, ignore_errors=True)
logger.info('Cleaned up old step/init checkpoints from Drive')

STAGE_CONFIGS = {
    1: 'training/stage1_layout_pretrain.yaml',
    2: 'training/stage2_doc_understanding.yaml',
    3: 'training/stage3_instruction_tuning.yaml',
}

def build_config(stage):
    with open(STAGE_CONFIGS[stage]) as f:
        cfg = yaml.safe_load(f)
    cfg['training']['batch_size'] = 4
    cfg['training']['gradient_accumulation_steps'] = 8
    cfg['training']['save_every_steps'] = 999999
    cfg['training']['eval_every_steps'] = 250
    cfg['training']['log_every_steps'] = 10
    cfg['training']['gradient_checkpointing'] = True
    cfg['training']['num_epochs'] = 1
    cfg['training']['learning_rate'] = float(cfg['training']['learning_rate'])
    cfg['training']['output_dir'] = f'/content/checkpoints/stage{stage}'
    cfg['data']['manifest_path'] = 'data/synthetic/output/manifest.jsonl'
    cfg['data']['stage'] = stage
    if stage > 1:
        cfg['model']['pretrained_checkpoint'] = f'/content/checkpoints/stage{stage-1}/best'
    return cfg

def run_stage(stage):
    logger.info(f'\n===== STAGE {stage} =====')
    cfg = build_config(stage)
    tc = cfg['training']
    local_dir = Path(tc['output_dir'])
    drive_dir = Path(f'{DRIVE_ROOT}/checkpoints/stage{stage}')
    drive_stage_file = drive_dir / 'stage_complete.txt'

    if drive_stage_file.exists():
        logger.info(f'Stage {stage} already complete on Drive. Skipping.')
        return

    mc = cfg['model']
    config = TinyDocVLMConfig(
        vision_config=mc.get('vision_config'),
        decoder_config=mc.get('decoder_config'),
        pixel_shuffle_scale=mc.get('pixel_shuffle_scale', 3),
        image_size=mc.get('image_size', 384),
        patch_size=mc.get('patch_size', 16),
    )
    model = TinyDocVLMForConditionalGeneration(config)

    pretrained = mc.get('pretrained_checkpoint')
    if pretrained and Path(pretrained).exists():
        logger.info(f'Loading pretrained from: {pretrained}')
        model = TinyDocVLMForConditionalGeneration.from_pretrained(pretrained)

    dc = cfg['data']
    dataset = DocumentDataset(
        data_root=dc.get('data_root', 'data/synthetic'),
        manifest_path=dc.get('manifest_path'),
        max_seq_length=dc.get('max_seq_length', 2048),
        stage=stage,
    )

    train_size = int(0.9 * len(dataset))
    eval_size = len(dataset) - train_size
    train_dataset, eval_dataset = random_split(dataset, [train_size, eval_size])
    logger.info(f'Dataset: {len(dataset)} total ({train_size} train / {eval_size} eval)')

    trainer_cfg = TrainerConfig(
        output_dir=str(local_dir),
        num_epochs=tc.get('num_epochs', 1),
        batch_size=tc.get('batch_size', 4),
        gradient_accumulation_steps=tc.get('gradient_accumulation_steps', 8),
        learning_rate=float(tc.get('learning_rate', 1e-4)),
        min_learning_rate=float(tc.get('min_learning_rate', 1e-5)),
        warmup_steps=100,
        weight_decay=tc.get('weight_decay', 0.01),
        max_grad_norm=tc.get('max_grad_norm', 1.0),
        max_seq_length=tc.get('max_seq_length', 2048),
        stage=stage,
        use_fp16=tc.get('use_fp16', True),
        save_every_steps=999999,
        eval_every_steps=tc.get('eval_every_steps', 250),
        log_every_steps=tc.get('log_every_steps', 10),
        gradient_checkpointing=tc.get('gradient_checkpointing', True),
        num_workers=1,
    )

    processor = TinyDocVLMProcessor()
    trainer = TinyDocVLMTrainer(
        model=model, processor=processor,
        train_dataset=train_dataset, eval_dataset=eval_dataset,
        config=trainer_cfg, device=device,
    )

    # Restore partial checkpoint from local or Drive
    if not (local_dir / 'trainer_state.pt').exists() and (drive_dir / 'trainer_state.pt').exists():
        logger.info('Restoring partial checkpoint from Drive...')
        if local_dir.exists():
            shutil.rmtree(str(local_dir))
        shutil.copytree(str(drive_dir), str(local_dir))

    if (local_dir / 'trainer_state.pt').exists():
        ckpts = sorted(local_dir.glob('epoch_*'))
        if ckpts:
            trainer.load_checkpoint(str(ckpts[-1]))
            logger.info(f'Resumed from {ckpts[-1]}')

    trainer.train()

    # Mark complete
    (local_dir / 'stage_complete.txt').write_text('done')

    # Clean local: keep only epoch_1 and best
    for d in local_dir.glob('init'):
        shutil.rmtree(d, ignore_errors=True)
    for d in sorted(local_dir.glob('step_*'))[:-1]:
        shutil.rmtree(d, ignore_errors=True)

    # Sync to Drive: overwrite old, keep ONLY this
    drive_dir.mkdir(parents=True, exist_ok=True)
    for item in drive_dir.iterdir():
        if item.is_dir():
            shutil.rmtree(str(item), ignore_errors=True)
        else:
            item.unlink()
    shutil.copytree(str(local_dir), str(drive_dir), dirs_exist_ok=True)
    logger.info(f'Stage {stage} synced to Drive')

for s in [1, 2, 3]:
    run_stage(s)

logger.info('\n===== ALL 3 STAGES COMPLETE! =====')
logger.info('Run Cell 5 to download the final model.')

## 5. Download final model

In [ ]:
from google.colab import files
import os

DRIVE_ROOT = '/content/drive/MyDrive/tinydoc-vlm'
final_dir = f'{DRIVE_ROOT}/checkpoints/stage3/best'

if os.path.exists(final_dir):
    !zip -r /content/tinydoc-vlm-final.zip {final_dir}
    files.download('/content/tinydoc-vlm-final.zip')
else:
    print('Stage 3 not complete yet.')